# 06a · Generate persona completions (vLLM) — per model → Drive

**Stage 1 of 2.** The *only* thing that gets generated in this whole pipeline: the persona-conditioned
completions that the CAA pathology vectors are built from. For each of 17 personas (10 clinical
mechanisms + 7 PC primitives) × 2 variants (persona-**on** vs neutral **baseline** system prompt) ×
~30 neutral prompts, the model writes a completion. `record = {mechanism, variant, prompt_idx,
prompt, sample, completion}`.

**Per-model, not frozen.** The *prompts* (neutral + persona system prompts) are shared across models,
but each organism generates its **own** completions — so the vector captures how *that* model expresses
the mechanism, on its own distribution. Only the weights + the resulting text differ.

vLLM is used purely for speed (it does **not** expose hidden states). Stage 2 (`06b`) re-loads each
model in plain HF transformers and reads activations off these saved completions. Output →
`DRIVE/directions_v1/generations_{clinical,pc}_{model}.jsonl`.

The desirability **probe** and desirability **steering vector** generate nothing (they read
task-prompt activations) — they live entirely in `06b`.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
import sys, subprocess, pathlib
PC = pathlib.Path("/content/Predictive_coding")
if not PC.exists():
    subprocess.check_call(["git","clone","https://github.com/ChuloIva/Predictive_coding.git", str(PC)])
LAB = PC / "steering_lab"
if str(LAB) not in sys.path: sys.path.insert(0, str(LAB))
print("steering on path:", (LAB/"steering"/"generate.py").exists())

In [ ]:
# vLLM for fast generation. Colab = CUDA 12.8 / torch cu128, but vLLM's default wheel +
# its "wheel-variant" auto-detection grab the cu13 build -> libcudart.so.13 missing
# (vllm#43435). Fix: install the EXPLICIT +cu129 wheel by direct URL (links libcudart.so.12,
# compatible with cu128 torch via CUDA-12 minor-version compat). Bypasses auto-detection.
import subprocess, sys, urllib.request, json, torch
print("driver CUDA (torch):", torch.version.cuda, "| torch:", torch.__version__)

def vllm_c_ext_ok():
    # verify in a CLEAN interpreter — this kernel may hold a half-imported cu13 vllm
    p = subprocess.run([sys.executable, "-c",
        "import vllm._C_stable_libtorch, vllm; print(vllm.__version__)"],
        capture_output=True, text=True)
    return p.returncode == 0, (p.stdout + p.stderr).strip()

ok, msg = vllm_c_ext_ok()
if ok:
    print("vllm", msg, "loads OK")
else:
    print("vllm unusable ->", msg.splitlines()[-1][:90], "\ninstalling +cu129 wheel...")
    # newest release tag (fallback to a known-good pin if the API is unreachable)
    try:
        tag = json.load(urllib.request.urlopen(
            "https://api.github.com/repos/vllm-project/vllm/releases/latest"))["tag_name"]
    except Exception:
        tag = "v0.24.0"
    ver = tag.lstrip("v")
    wheel = (f"https://github.com/vllm-project/vllm/releases/download/{tag}/"
             f"vllm-{ver}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl")
    print("installing:", wheel)
    !pip -q uninstall -y vllm
    # keep Colab's torch 2.11.0 (pin is satisfied); cu129 extra-index steers any cuda deps to cu12
    !pip install -q "{wheel}" --extra-index-url https://download.pytorch.org/whl/cu129
    ok, msg = vllm_c_ext_ok()
    assert ok, f"vllm +cu129 install still broken:\n{msg}"
    print("vllm", msg, "installed OK.  If this kernel already imported the old vllm, "
          "do Runtime > Restart, then re-run from here.")

%pip install -q -U transformers accelerate sentencepiece
import importlib; importlib.import_module("transformers"); print("transformers ok")

# vLLM's suppress_stdout() calls sys.stdout.fileno(); ipykernel's stream has no real fd
# -> "io.UnsupportedOperation: fileno" when EngineCore starts. Give the streams a real fd
# (this patch survives the fork into the EngineCore subprocess, which inherits sys.stdout).
def _ensure_fileno(stream, fd):
    try:
        stream.fileno(); return
    except Exception:
        try: stream.fileno = lambda: fd
        except Exception: pass
_ensure_fileno(sys.stdout, 1); _ensure_fileno(sys.stderr, 2)
print("stdout.fileno ->", sys.stdout.fileno())

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for public repos)")

In [ ]:
DRIVE = mount_drive()
assert DRIVE is not None, "Drive not mounted — generations must persist for 06b"
OUT = DRIVE / "directions_v1"; OUT.mkdir(parents=True, exist_ok=True)
print("generations ->", OUT)

## 2. Config — models + generation params

In [ ]:
import json
MODELS = json.load(open("/content/dt_rl/notebooks/organisms.json"))["models"]  # edit HF ids THERE
print("organisms:", ", ".join(f"{m['name']}{'' if m['hf'] else '(-)'}" for m in MODELS))

USE_VLLM      = True   # False -> HF-transformers generation (no vLLM install needed)
CAA_NEUTRAL_N = 30     # neutral prompts per persona/variant (None = all)
CAA_MAXTOK    = 160    # completion length
GEN_TEMP      = 0.8
GEN_SEED      = 0
CAA_GEN_BATCH = 8      # HF-fallback batch only
print(f"{sum(1 for m in MODELS if m['hf'])}/{len(MODELS)} checkpoints; vLLM={USE_VLLM}")

## 3. Generate — per model, clinical + PC sets

In [ ]:
from steering import config, generate
from steering.personas import PERSONAS, NEUTRAL_PROMPTS
from steering.personas_pc import PC_PERSONAS
import gc, torch

NEUTRAL = NEUTRAL_PROMPTS if CAA_NEUTRAL_N is None else NEUTRAL_PROMPTS[:CAA_NEUTRAL_N]
CAA_SETS = [("clinical", PERSONAS), ("pc", PC_PERSONAS)]

def free_vllm(llm):
    try:
        from vllm.distributed.parallel_state import (destroy_model_parallel,
                                                      destroy_distributed_environment)
        destroy_model_parallel(); destroy_distributed_environment()
    except Exception: pass
    del llm; gc.collect(); torch.cuda.empty_cache()

for spec in MODELS:
    name, hf = spec["name"], spec["hf"]
    if not hf:
        print(f"skip {name}: no checkpoint"); continue
    print(f"\n{'='*60}\n{name} :: {hf}\n{'='*60}")
    gcfg = config.GenConfig(model_name=hf, max_tokens=CAA_MAXTOK, temperature=GEN_TEMP, seed=GEN_SEED)
    llm = tok = None
    if USE_VLLM:
        try:
            from vllm import LLM
            from steering.steer import _load_tokenizer
            tok = _load_tokenizer(hf, os.environ.get("HF_TOKEN"))
            llm = LLM(model=hf, dtype="bfloat16", max_model_len=gcfg.max_model_len,
                      gpu_memory_utilization=gcfg.gpu_memory_utilization)
        except Exception as e:
            print(f"[{name}] vLLM load failed ({type(e).__name__}: {str(e)[:120]}) -> HF fallback")
            llm = None
    for set_name, personas in CAA_SETS:
        out_path = OUT / f"generations_{set_name}_{name}.jsonl"
        print(f"[{name}/{set_name}] {len(personas)}x2x{len(NEUTRAL)} completions -> {out_path.name}")
        if llm is not None:
            generate.generate_dataset(gcfg, prompts=NEUTRAL, personas=personas,
                                      out_path=str(out_path), llm=llm, tokenizer=tok)
        else:
            generate.generate_dataset_hf(gcfg, prompts=NEUTRAL, personas=personas,
                                         out_path=str(out_path), batch_size=CAA_GEN_BATCH)
    if llm is not None: free_vllm(llm)
    gc.collect(); torch.cuda.empty_cache()
    print(f"[{name}] done. GPU:", round(torch.cuda.memory_allocated()/1e9,2), "GB")
print("\nAll generations saved. If model #2+ OOMs on vLLM reload, set MODELS to one model, "
      "Runtime->Restart, and run per model (or USE_VLLM=False).")

## Output
`DRIVE/directions_v1/generations_clinical_<model>.jsonl` and `generations_pc_<model>.jsonl` — one
JSONL per (set, model), first line a `_config` header, then `{mechanism, variant, prompt_idx, prompt,
sample, completion}` rows. These are also eyeball-able Tier-0 behavioural data (do the personas
actually work?). Next: **`06b`** reads activations off these and extracts all directions.